<a href="https://colab.research.google.com/github/zombie9088/Pytorch_learning/blob/main/Rnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("/content/drive/MyDrive/fmnist/100_Unique_QA_Dataset.csv")
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [3]:
#tokenize
def tokenize(text):
  text=text.lower()
  text=text.replace('?','')
  text=text.replace("'",'')
  return text.split()

In [4]:
tokenize("my name is swap")

['my', 'name', 'is', 'swap']

In [5]:
#vocab
vocab={'<UNK>':0}


In [6]:
def build_vocab(row):
  print(row['question'],row['answer'])
  tokenized_question=tokenize(row['question'])
  tokenized_answer=tokenize(row['answer'])
  merged = tokenized_question + tokenized_answer

  for token in merged:
    if token not in vocab:
      vocab[token]=len(vocab)


In [7]:
df.apply(build_vocab,axis=1)

What is the capital of France? Paris
What is the capital of Germany? Berlin
Who wrote 'To Kill a Mockingbird'? Harper-Lee
What is the largest planet in our solar system? Jupiter
What is the boiling point of water in Celsius? 100
Who painted the Mona Lisa? Leonardo-da-Vinci
What is the square root of 64? 8
What is the chemical symbol for gold? Au
Which year did World War II end? 1945
What is the longest river in the world? Nile
What is the capital of Japan? Tokyo
Who developed the theory of relativity? Albert-Einstein
What is the freezing point of water in Fahrenheit? 32
Which planet is known as the Red Planet? Mars
Who is the author of '1984'? George-Orwell
What is the currency of the United Kingdom? Pound
What is the capital of India? Delhi
Who discovered gravity? Newton
How many continents are there on Earth? 7
Which gas do plants use for photosynthesis? CO2
What is the smallest prime number? 2
Who invented the telephone? Alexander-Graham-Bell
What is the capital of Australia? Canber

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [8]:

len(vocab)

324

In [9]:
#convert words to indices

def text_to_indices(vocab,text):
  indexed_text= []

  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])
  return indexed_text

In [10]:
text_to_indices(vocab, "what is gayaf ")

[1, 2, 0]

In [11]:
import torch
from torch.utils.data import Dataset , DataLoader

In [22]:
class QADataset(Dataset):
  def __init__(self,df,vocab):
    self.df=df
    self.vocab=vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self,idx):
    # Fix: Swapped the arguments for text_to_indices
    num_question=text_to_indices(self.vocab, self.df.iloc[idx]['question'])
    num_answer=text_to_indices(self.vocab, self.df.iloc[idx]['answer'])

    return torch.tensor(num_question),torch.tensor(num_answer)

In [23]:
dataset= QADataset(df,vocab)

In [24]:
dataset[10]

(tensor([ 1,  2,  3,  4,  5, 53]), tensor([54]))

In [21]:
dataloader= DataLoader(dataset,batch_size=1,shuffle=True)

In [30]:
import torch.nn as nn

class RNNModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super(RNNModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True) # batch_first=True for (batch_size, seq_len, features)
        self.fc = nn.Linear(hidden_size, vocab_size) # Output layer to predict next token in vocab

    def forward(self, x):
        # x shape: (batch_size, seq_len)
        embedded = self.embedding(x)
        # embedded shape: (batch_size, seq_len, embedding_dim)

        output, hidden = self.rnn(embedded)
        # output shape: (batch_size, seq_len, hidden_size)
        # hidden shape: (1, batch_size, hidden_size) for single layer RNN

        # Apply linear layer to the output (e.g., to predict next word for each position)
        logits = self.fc(output)
        # logits shape: (batch_size, seq_len, vocab_size)
        return logits, hidden

# Instantiate the model
embedding_dim = 50
hidden_size = 64
vocab_size = len(vocab) # Assuming 'vocab' is defined and contains all unique tokens

model = RNNModel(vocab_size, embedding_dim, hidden_size)
print(model)

RNNModel(
  (embedding): Embedding(324, 50)
  (rnn): RNN(50, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=324, bias=True)
)


In [31]:
import torch.optim as optim

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=vocab['<UNK>']) # Ignore UNK token for loss calculation
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training hyperparameters
num_epochs = 50

print("Starting training loop...")

for epoch in range(num_epochs):
    model.train() # Set the model to training mode
    total_loss = 0

    for batch_idx, (question_tensor, answer_tensor) in enumerate(dataloader):
        optimizer.zero_grad() # Zero the gradients

        # Ensure tensors have a batch dimension. Dataloader usually provides this
        # for batch_size > 0, so these unsqueeze calls might not always be necessary
        # but are kept for robustness if data loading changes.
        if question_tensor.dim() == 1: # If only sequence length is present, add batch dim
            question_tensor = question_tensor.unsqueeze(0)
        if answer_tensor.dim() == 1: # Same for answer_tensor
            answer_tensor = answer_tensor.unsqueeze(0)

        # Forward pass
        output_logits, _ = model(question_tensor)
        # output_logits shape: (batch_size, seq_len_question, vocab_size)

        # --- FIX: Align model output with target for CrossEntropyLoss ---
        # For a simplified setup, we'll try to predict the first answer token
        # using the output corresponding to the last token of the question sequence.
        # This is a common simplification for initial RNN baselines.

        # Take the output logits for the last token of the input question sequence
        # Shape becomes (batch_size, vocab_size)
        predicted_answer_logits = output_logits[:, -1, :]

        # Take the first token of the actual answer sequence as the target
        # Shape becomes (batch_size,)
        target_answer_token = answer_tensor[:, 0]

        # Calculate loss
        # CrossEntropyLoss expects input (N, C) and target (N)
        loss = criterion(predicted_answer_logits, target_answer_token)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

print("Training complete.")

Starting training loop...
Epoch 1/50, Loss: 5.8619
Epoch 2/50, Loss: 5.1274
Epoch 3/50, Loss: 4.2483
Epoch 4/50, Loss: 3.5304
Epoch 5/50, Loss: 2.9520
Epoch 6/50, Loss: 2.4131
Epoch 7/50, Loss: 1.9278
Epoch 8/50, Loss: 1.5022
Epoch 9/50, Loss: 1.1527
Epoch 10/50, Loss: 0.8830
Epoch 11/50, Loss: 0.6862
Epoch 12/50, Loss: 0.5416
Epoch 13/50, Loss: 0.4288
Epoch 14/50, Loss: 0.3508
Epoch 15/50, Loss: 0.2892
Epoch 16/50, Loss: 0.2421
Epoch 17/50, Loss: 0.2039
Epoch 18/50, Loss: 0.1749
Epoch 19/50, Loss: 0.1511
Epoch 20/50, Loss: 0.1304
Epoch 21/50, Loss: 0.1137
Epoch 22/50, Loss: 0.1008
Epoch 23/50, Loss: 0.0886
Epoch 24/50, Loss: 0.0792
Epoch 25/50, Loss: 0.0714
Epoch 26/50, Loss: 0.0641
Epoch 27/50, Loss: 0.0578
Epoch 28/50, Loss: 0.0525
Epoch 29/50, Loss: 0.0478
Epoch 30/50, Loss: 0.0437
Epoch 31/50, Loss: 0.0403
Epoch 32/50, Loss: 0.0371
Epoch 33/50, Loss: 0.0342
Epoch 34/50, Loss: 0.0316
Epoch 35/50, Loss: 0.0294
Epoch 36/50, Loss: 0.0273
Epoch 37/50, Loss: 0.0254
Epoch 38/50, Loss: 0.

In [38]:
def predict_answer(question_text, model, vocab):
    model.eval() # Set the model to evaluation mode
    with torch.no_grad(): # Disable gradient calculation for inference
        # Convert question text to numerical indices
        question_indices = text_to_indices(vocab, question_text)
        if not question_indices:
            return "I have no Idea" # Handle empty or unresolvable questions

        # Convert indices to a tensor and add a batch dimension
        question_tensor = torch.tensor(question_indices).unsqueeze(0)

        # Forward pass through the model
        output_logits, _ = model(question_tensor)

        # Take the logits corresponding to the last token of the input question sequence
        # This is based on the training objective to predict the first answer token
        # from the last output of the question sequence.
        predicted_token_logits = output_logits[:, -1, :]

        # Get the predicted token index (index with highest probability)
        predicted_token_idx = torch.argmax(predicted_token_logits, dim=-1).item()

        # Reverse lookup to get the word from the index
        # Create an inverted vocab for easy lookup
        inverted_vocab = {v: k for k, v in vocab.items()}
        predicted_answer_word = inverted_vocab.get(predicted_token_idx, "<UNK>")

        return predicted_answer_word




In [41]:
question = "where is bandel"
predicted_word = predict_answer(question, model, vocab)
print(f"Question: {question}\nPredicted Answer: {predicted_word}")

Question: where is bandel
Predicted Answer: pacific-ocean
